# ENVRI HUB library exercise
In this notebook you'll learn how to use the *VRE-Lib* component developed by WP 13 and WP 14.
The VRE-Lib is available as a *Python package* on the [public package index](https://pypi.org/project/envrihub/) un the *envrihub* name.
This means it can be installed with:

In [ ]:
! pip install --upgrade envrihub==0.1.3
! pip install openapi-spec-validator

The main element you have to import is the _Hub_ object:

In [ ]:
from envrihub import Hub

hub = Hub()

## Exercise
Use the _Hub_ object to query the Catalogue of Services and retrieve data for a specific Essential Variable: **Ocean Temperature**. Remember you can always ask the [Environmental Expert](https://chat.envri.eu/) for help. The steps of the exercise are:
+ find a service providing data for Ocean Temperature;
+ access the service and retrieve data;
+ display data on a table and/or a graph.

In [ ]:
for res in hub.search_catalogue('ocean temperature'):
    print(res.title)
    print(f'\t resource id: {res.uid}')
    print(f'\t resource type: {res.type}')
    print(f'\t resurce description: {res.description}\n\n')

In [ ]:
service_id = 'file:///ArgoFloats_distribution_003'

res = hub.fetch_from_catalogue(service_id)
print(res.title)

Where do you get IDs? Either browsing the catlogue on your own, or with the `search_catalogue` method.

What we have now inside the `res` variable is a `Distribution` object that has the following attributes:
+ `title`: the resource title/display name
+ `uid`: the unique internal identifier to fetch it a later time
+ `description`: a human readable minimal description
+ `type`: whether it is a web service or a file
+ `href`: a link to more metadata
+ `service_documentation`: a link to some material that should allow you to get a hold on how to use the resouce.
+ `metadata`: all the metadata avaiable.

Plus the `is_downloadable` function that answers the fundamental question for all you digital kleptomaniacs: *can I download it on my laptop?*

In [ ]:
res.metadata

Quite some information isn't it?
That's becasue you might want to know what you'll find inside the data *before* opening it. You know... To avoid jumpscares.

Now it's finally time to access the *actual data* with the `dao` attribute, that *always* comes with helpful documentation you can access with the *help()* function or with third party Jupyter extensions.

In [ ]:
catalogue_dao = res.dao
help(catalogue_dao)

And now let's do some magic with data. The [Environmental Expert](https://chat.envri.eu/) can help you with this.

In [ ]:
import pandas as pd
import json

service_response = catalogue_dao.access(latitude_less='33', latitude_more='29', longitude_less='30', longitude_more='-21', pres_less='10', time_less='2025-02-01T00:00:00Z', pres_more='0', time_more='2025-01-01T00:00:00Z')

# 1. The service returns a geoJSON, let's load it
data = json.loads(service_response)

# 2. Extract features
features = data['features']

# 3. Transform 'properties' into dictionaries

rows = []
for f in features:
    # Get all properties (temp, pres, psal, ecc.)
    record = f['properties'].copy()
    
    # Extract coordinates: [longitudine, latitudine]
    record['longitude'] = f['geometry']['coordinates'][0]
    record['latitude'] = f['geometry']['coordinates'][1]
    
    rows.append(record)

# 4. Create DataFrame
df = pd.DataFrame(rows)

# 5. Remove empty column
df = df.replace("", float('nan'))
df_pulito = df.dropna(axis=1, how='all')

# Print result
print(f"Clean table: {df_pulito.shape[0]} rows and {df_pulito.shape[1]} columns.")
print(df_pulito.head())